# 04 Error Correction Model
Estimate ECM and run diagnostic tests.

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_white, acorr_breusch_godfrey
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path
ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data/processed/moutai_profit_processed.csv")

In [ ]:
long_run = sm.OLS(df["LNY"], sm.add_constant(df[["LNX2","LNX3","LNX4","LNX5"]]), missing="drop").fit()
df["ECM"] = long_run.resid
df["ECM_L1"] = df["ECM"].shift(1)
model_df = df.dropna(subset=["DLNY","DLNX2","DLNX3","DLNX4","DLNX5","ECM_L1"])
ecm = sm.OLS(model_df["DLNY"], sm.add_constant(model_df[["DLNX2","DLNX3","DLNX4","DLNX5","ECM_L1"]])).fit()
print(ecm.summary())

In [ ]:
x = sm.add_constant(model_df[["DLNX2","DLNX3","DLNX4","DLNX5","ECM_L1"]])
print([variance_inflation_factor(x.values, i) for i in range(x.shape[1])])
print("White:", het_white(ecm.resid, ecm.model.exog))
print("BG LM:", acorr_breusch_godfrey(ecm, nlags=2))